# 01 — EDA & Data Pipeline
**Plant Disease Detection (Residual CBM)** — Step 1 of 4

Self-contained notebook: every function used below is defined in this
notebook (no `scripts/` import, no `sys.path` manipulation) so it runs
identically regardless of which directory the kernel starts in — Jupyter,
VS Code, Colab, or Antigravity.

Builds the dataset manifest (single source of truth for the 70/15/15
train/val/test split), explores class balance, visualizes samples, and runs
a perceptual-hash leakage check before any model sees the data.

**Prerequisites:** raw PlantVillage images extracted under `data/raw/`.


In [ ]:
# --- Standard imports ---
import os
import ast

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from sklearn.model_selection import train_test_split
import imagehash

pd.set_option("display.max_columns", 20)
sns.set_style("whitegrid")

print(f"[INFO] pandas {pd.__version__} | numpy {np.__version__}")


In [ ]:
# --- Dataset constants (kept identical across all 4 notebooks and to the
# original manifest.csv build, so label indices / concept vectors never
# drift between notebooks or against the already-trained model) ---
TARGET_SPECIES = ["Potato", "Tomato", "Pepper"]
MAX_IMAGES_PER_CLASS = 200
IMG_SIZE = 224
SEED = 42

SELECTED_CLASSES = [
    "Pepper__bell___Bacterial_spot", "Pepper__bell___healthy", "Potato___Early_blight",
    "Potato___Late_blight", "Potato___healthy", "Tomato_Bacterial_spot",
    "Tomato_Early_blight", "Tomato_Late_blight", "Tomato_Leaf_Mold",
    "Tomato_Septoria_leaf_spot", "Tomato_Spider_mites_Two_spotted_spider_mite",
    "Tomato__Target_Spot", "Tomato__Tomato_YellowLeaf__Curl_Virus",
    "Tomato__Tomato_mosaic_virus", "Tomato_healthy",
]
LABEL_MAP = {k: i for i, k in enumerate(SELECTED_CLASSES)}

CONCEPT_NAMES = ["Is Healthy", "Has Spots", "Is Blighted", "Complex Pathogen"]
CONCEPT_MAP = {
    "Pepper__bell___Bacterial_spot":               [0.0, 1.0, 0.0, 0.0],
    "Pepper__bell___healthy":                      [1.0, 0.0, 0.0, 0.0],
    "Potato___Early_blight":                       [0.0, 0.0, 1.0, 0.0],
    "Potato___Late_blight":                        [0.0, 0.0, 1.0, 0.0],
    "Potato___healthy":                            [1.0, 0.0, 0.0, 0.0],
    "Tomato_Bacterial_spot":                       [0.0, 1.0, 0.0, 0.0],
    "Tomato_Early_blight":                         [0.0, 0.0, 1.0, 0.0],
    "Tomato_Late_blight":                          [0.0, 0.0, 1.0, 0.0],
    "Tomato_Leaf_Mold":                            [0.0, 0.0, 0.0, 1.0],
    "Tomato_Septoria_leaf_spot":                   [0.0, 1.0, 0.0, 0.0],
    "Tomato_Spider_mites_Two_spotted_spider_mite": [0.0, 1.0, 0.0, 0.0],
    "Tomato__Target_Spot":                         [0.0, 1.0, 0.0, 0.0],
    "Tomato__Tomato_YellowLeaf__Curl_Virus":       [0.0, 0.0, 0.0, 1.0],
    "Tomato__Tomato_mosaic_virus":                 [0.0, 0.0, 0.0, 1.0],
    "Tomato_healthy":                              [1.0, 0.0, 0.0, 0.0],
}
# NOTE (honesty flag): these 4 "concepts" are a deterministic re-encoding of
# the 15 class labels, not independently annotated ground truth -- every
# image of a class gets an identical concept vector. Valid auxiliary
# hierarchical-label signal, not a from-scratch Concept Bottleneck Model in
# the Koh et al. sense.

print(f"[INFO] Target species: {TARGET_SPECIES}")
print(f"[INFO] Classes tracked: {len(SELECTED_CLASSES)}")
print(f"[INFO] Per-class undersampling cap: {MAX_IMAGES_PER_CLASS}")


## 1.1 Locate the raw dataset

In [ ]:
candidate_paths = ["data/raw/PlantVillage", "data/raw/plantvillage dataset/color", "data/raw/plantvillage_dataset/color",
                   "../data/raw/PlantVillage", "../data/raw/plantvillage dataset/color"]
base_path = next((p for p in candidate_paths if os.path.isdir(p)), None)

if base_path is None:
    raise FileNotFoundError(
        "Could not find the extracted PlantVillage folder under data/raw/ "
        "(checked both the project root and one level up, in case the kernel's "
        "working directory is notebooks/). Run scripts/kaggle_download.py first."
    )

print(f"[OK] Using raw dataset at: {base_path}")
print(f"[INFO] Top-level entries found: {len(os.listdir(base_path))}")


## 1.2 Build the raw dataframe
Walks the class folders and records every image path — no filtering yet, just an inventory.

In [ ]:
def build_dataframe(base_path):
    all_folders = os.listdir(base_path)
    selected = [f for f in all_folders if any(s in f for s in TARGET_SPECIES)]
    rows = []
    for c in selected:
        folder_path = os.path.join(base_path, c)
        if not os.path.isdir(folder_path):
            continue
        for img in os.listdir(folder_path):
            if img.lower().endswith((".jpg", ".jpeg")):
                rows.append({"File_Name": img, "Class_Name": c, "File_Path": os.path.join(folder_path, img)})
    return pd.DataFrame(rows)

raw_df = build_dataframe(base_path)

print(f"[RESULT] Raw images found: {len(raw_df):,}")
print(f"[RESULT] Classes represented: {raw_df['Class_Name'].nunique()}")
print()
print("Per-class raw image counts (top 15):")
print(raw_df["Class_Name"].value_counts().head(15))


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
raw_df["Class_Name"].value_counts().sort_values().plot.barh(ax=ax, color="#4C72B0")
ax.set_title("Raw image count per class (pre-undersampling)")
ax.set_xlabel("Image count")
plt.tight_layout()
plt.show()


## 1.3 Undersample & stratified 70/15/15 split

In [ ]:
def undersample(df, cap=MAX_IMAGES_PER_CLASS, seed=SEED):
    return (
        df.groupby("Class_Name")
        .apply(lambda x: x.sample(n=min(len(x), cap), random_state=seed), include_groups=False)
        .reset_index()
    )

balanced_df = undersample(raw_df)
print(f"[RESULT] After undersampling: {len(balanced_df):,} images")
print(balanced_df["Class_Name"].value_counts().describe())


In [ ]:
def stratified_split(df, seed=SEED):
    labels = df["Class_Name"].map(LABEL_MAP)
    train_df, temp_df = train_test_split(df, test_size=0.30, stratify=labels, random_state=seed)
    temp_labels = temp_df["Class_Name"].map(LABEL_MAP)
    val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_labels, random_state=seed)
    train_df = train_df.copy(); train_df["split"] = "train"
    val_df = val_df.copy(); val_df["split"] = "val"
    test_df = test_df.copy(); test_df["split"] = "test"
    return pd.concat([train_df, val_df, test_df], ignore_index=True)

split_df = stratified_split(balanced_df)

print("[RESULT] Rows per split:")
print(split_df.groupby("split").size())
print()
print("[RESULT] Class balance is preserved across splits (crosstab):")
print(pd.crosstab(split_df["Class_Name"], split_df["split"]))


In [ ]:
def export_manifest(df, out_csv="../manifest.csv"):
    out = df.copy()
    out["label_idx"] = out["Class_Name"].map(LABEL_MAP)
    out["concept_vec"] = out["Class_Name"].map(lambda c: CONCEPT_MAP[c])
    out.to_csv(out_csv, index=False)
    return out_csv

manifest_csv_path = "manifest.csv" if os.path.isdir("data") else "../manifest.csv"
manifest_path = export_manifest(split_df, out_csv=manifest_csv_path)
manifest = pd.read_csv(manifest_path)
manifest["concept_vec"] = manifest["concept_vec"].apply(ast.literal_eval)

print(f"[SUCCESS] manifest.csv written to {manifest_path} with shape {manifest.shape}")
print(manifest[["Class_Name", "split", "label_idx"]].head())


## 1.4 Visualize sample images per split

In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(15, 9))
for row_i, split_name in enumerate(["train", "val", "test"]):
    sample = manifest[manifest["split"] == split_name].sample(5, random_state=42)
    for col_i, (_, r) in enumerate(sample.iterrows()):
        ax = axes[row_i, col_i]
        try:
            with Image.open(r["File_Path"]) as img:
                ax.imshow(img)
        except Exception:
            ax.text(0.5, 0.5, "unreadable", ha="center")
        ax.set_title(f"{split_name}\n{r['Class_Name']}", fontsize=8)
        ax.axis("off")
plt.tight_layout()
plt.show()
print("[INFO] 5 random samples shown per split (train/val/test), same manifest used downstream.")


## 1.5 Cross-split leakage check
Perceptual-hashes every image and flags any pair landing in *different* splits
within a small hamming distance — near-duplicates crossing the split boundary
would silently inflate reported test/val accuracy.

In [ ]:
def hash_all(df, hash_size=8):
    hashes = {}
    for _, row in df.iterrows():
        try:
            with Image.open(row["File_Path"]) as img:
                hashes[row["File_Path"]] = imagehash.phash(img, hash_size=hash_size)
        except Exception as e:
            print(f"[WARN] could not hash {row['File_Path']}: {e}")
    return hashes

def find_cross_split_duplicates(df, hashes, threshold=5):
    paths = list(hashes.keys())
    split_by_path = dict(zip(df["File_Path"], df["split"]))
    class_by_path = dict(zip(df["File_Path"], df["Class_Name"]))
    flagged = []
    for i in range(len(paths)):
        for j in range(i + 1, len(paths)):
            p1, p2 = paths[i], paths[j]
            if split_by_path[p1] == split_by_path[p2]:
                continue
            dist = hashes[p1] - hashes[p2]
            if dist <= threshold:
                flagged.append({"path_a": p1, "split_a": split_by_path[p1], "class_a": class_by_path[p1],
                                 "path_b": p2, "split_b": split_by_path[p2], "class_b": class_by_path[p2],
                                 "hash_distance": dist})
    return pd.DataFrame(flagged)

print(f"[OPERATION] Hashing {len(manifest)} images for leakage detection (this can take a few minutes)...")
hashes = hash_all(manifest)
dup_df = find_cross_split_duplicates(manifest, hashes, threshold=5)
dup_out_path = "duplicate_report.csv" if os.path.isdir("data") else "../duplicate_report.csv"
dup_df.to_csv(dup_out_path, index=False)

if len(dup_df) == 0:
    print("[RESULT] No cross-split near-duplicates found at threshold=5.")
else:
    n_test = dup_df[dup_df["split_a"].eq("test") | dup_df["split_b"].eq("test")].shape[0]
    print(f"[FLAG] {len(dup_df)} cross-split near-duplicate pairs found ({n_test} touch the test split).")
    print(dup_df.head())


## 1.6 Summary

| Check | Result |
|---|---|
| Classes tracked | 15 (Potato / Tomato / Pepper subset) |
| Split ratio | 70 / 15 / 15, stratified |
| Manifest | `manifest.csv` — single source of truth read by every downstream notebook |
| Leakage | see `duplicate_report.csv` printed above |

**Next notebook:** `02_Model_Training.ipynb` reads `manifest.csv` directly.
